# **Batsman Workflow**

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [2]:
class ScoreStats(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percentage: float
    summary: str

In [3]:
def strike_rate(state: ScoreStats):
    sr = (state['runs'] / state['balls']) * 100
    return {'sr': sr}


In [4]:
def bpb(state: ScoreStats):
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'bpb': bpb}

In [5]:
def boundary_percentage(state: ScoreStats):
    boundary_percentage = (((state['fours'] * 4) + (state['sixes'] * 6))/state['runs'])*100
    return {'boundary_percentage': boundary_percentage}

In [6]:
def summary(state: ScoreStats):
    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percentage - {state['boundary_percentage']}
"""
    
    return {'summary': summary}

In [7]:
# Graph
graph = StateGraph(ScoreStats)

# Nodes
graph.add_node('strike_rate', strike_rate)
graph.add_node('bpb', bpb)
graph.add_node('boundary_percentage', boundary_percentage)
graph.add_node('summary', summary)

# Parallel Edges
graph.add_edge(START, 'strike_rate')
graph.add_edge(START, 'bpb')
graph.add_edge(START, 'boundary_percentage')

graph.add_edge('strike_rate','summary')
graph.add_edge('bpb','summary')
graph.add_edge('boundary_percentage','summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [8]:
# Execute 
initial_state = {'runs':230, 'balls':93, 'fours':15, 'sixes': 11}

final_state = workflow.invoke(initial_state)

In [9]:
print(final_state)

{'runs': 230, 'balls': 93, 'fours': 15, 'sixes': 11, 'sr': 247.31182795698925, 'bpb': 3.576923076923077, 'boundary_percentage': 54.78260869565217, 'summary': '\nStrike Rate - 247.31182795698925 \n\nBalls per boundary - 3.576923076923077 \n\nBoundary percentage - 54.78260869565217\n'}


# **UPSC Essay Evaluation Workflow**

In [10]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [11]:
load_dotenv()

model = ChatOpenAI(model = 'gpt-4o-mini')

In [12]:
# define output schema
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [13]:
# Model with structured output
structured_model = model.with_structured_output(EvaluationSchema)

In [14]:
class EvalStats(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores : Annotated[list[int], operator.add] # Reducer - receive the invidual scores and store in list format
    avg_score: float

In [15]:
def evaluate_language(state: EvalStats):
    prompt = f"Evaluate the Language Quality of the following essay and assign a score out of 10 \n {state['essay']}"

    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [16]:
def evaluate_analysis(state: EvalStats):
    prompt = f"Evaluate the Depth of Analysis of the following essay and assign a score out of 10 \n {state['essay']}"

    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [17]:
def evaluate_clarity(state: EvalStats):
    prompt = f"Evaluate the Clarity of Thought of the following essay and assign a score out of 10 \n {state['essay']}"

    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [18]:
def evaluate_ovoral(state: EvalStats):
    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [19]:
# Graph
graph = StateGraph(EvalStats)

graph.add_node('Language Quality', evaluate_language)
graph.add_node('Depth of Analysis', evaluate_analysis)
graph.add_node('Clarity of Thought', evaluate_clarity)
graph.add_node('Ovoral Evaluation', evaluate_ovoral)

# Parallel Edges
graph.add_edge(START, 'Language Quality')
graph.add_edge(START, 'Depth of Analysis')
graph.add_edge(START, 'Clarity of Thought')

graph.add_edge('Language Quality', 'Ovoral Evaluation')
graph.add_edge('Depth of Analysis', 'Ovoral Evaluation')
graph.add_edge('Clarity of Thought', 'Ovoral Evaluation')

graph.add_edge('Ovoral Evaluation', END)

workflow = graph.compile()

In [20]:
essay1 = """
Title - India in the Age of AI

As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society.

"""

In [21]:
initial_state = {'essay': essay1}

final_state = workflow.invoke(initial_state)

In [23]:
print("\n ---Language Quality Feedback--- \n",final_state['language_feedback'])
print("\n ---Depth of Analysis Feedback--- \n",final_state['analysis_feedback'])
print("\n ---Clarity of Thought Feedback--- \n",final_state['clarity_feedback'])
print("\n ---Overall Feedback--- \n",final_state['overall_feedback'])
print("\n ---Individual Scores--- \n",final_state['individual_scores'])
print("\n ---Avrage Score--- \n",final_state['avg_score'])


 ---Language Quality Feedback--- 
 The essay on 'India in the Age of AI' provides a thorough exploration of both the opportunities and challenges facing India in the realm of artificial intelligence. The structure of the essay is logical, moving from India's strengths to the applications of AI, and then addressing potential challenges before concluding with a forward-looking vision which is commendable. 

Linguistically, the essay is well-crafted, with a good command of vocabulary and a varied sentence structure that enhances readability. Phrases such as "critical juncture" and "socio-economic inequalities" show a sophisticated use of language, fitting for an academic discussion. The use of specific examples like predictive analytics in agriculture and AI in healthcare demonstrates a strong understanding of the subject matter. 

However, the essay could improve by ensuring that it includes more concrete statistical evidence to support claims made, particularly in areas discussing job 